# Microsoft Agent Framework - Complete Tutorial

**Version:** 0.1.0-beta  
**Last Updated:** February 18, 2026 - 7:50 AM ET  
**Duration:** 45-60 minutes

This tutorial covers advanced Agent Framework concepts:
1. Building specialized agents
2. Multi-agent workflows with graphs
3. Thread management and state
4. Tool integration patterns
5. Conditional routing and parallel execution
6. Error handling and observability

## Setup

In [ ]:
# Imports
import os
import sys
import asyncio
import json
from pathlib import Path
from typing import Dict, List, Any

# Setup paths
foundry_path = Path.cwd().parent
sys.path.insert(0, str(foundry_path))

# Load environment
from dotenv import load_dotenv
load_dotenv(foundry_path / ".env")

# Azure authentication
from azure.identity import DefaultAzureCredential
credential = DefaultAzureCredential()

print("✅ Setup complete")

## 1. Building Specialized Agents

Different tasks require different agent configurations.

In [ ]:
from agent_framework import Agent

# Analyst Agent: High precision, factual
analyst = Agent(
    name="PolicyAnalyst",
    instructions="""You are an expert EI policy analyst.
    
    Your responsibilities:
    - Analyze policy questions with precision
    - Cite specific sections of EI Act and Regulations
    - Identify relevant precedents
    - Provide clear, structured analysis
    
    Always be accurate and cite sources.""",
    model="gpt-4o",
    temperature=0.2  # Low for factual accuracy
)

# Writer Agent: Clear explanations
writer = Agent(
    name="PlainLanguageWriter",
    instructions="""You explain complex EI policies in plain language.
    
    Your responsibilities:
    - Take technical policy analysis
    - Rewrite for general public (Grade 8 reading level)
    - Use examples and analogies
    - Maintain accuracy while simplifying
    
    Be clear, friendly, and helpful.""",
    model="gpt-4o",
    temperature=0.6  # Higher for creative explanations
)

# Calculator Agent: Precise calculations
calculator = Agent(
    name="BenefitCalculator",
    instructions="""You calculate EI benefit amounts accurately.
    
    Your responsibilities:
    - Calculate weekly benefit amount (55% of average earnings)
    - Apply maximum weekly benefit ($668 in 2026)
    - Determine benefit duration based on hours and region
    - Show all calculation steps
    
    Double-check all math.""",
    model="gpt-4o",
    temperature=0.1  # Lowest for calculations
)

print("✅ Three specialized agents created")

In [ ]:
# Test each agent individually
async def test_agents():
    # Analyst
    analysis = await analyst.run(
        "What are the hour requirements for EI regular benefits?",
        thread_id="analyst-demo"
    )
    print("📊 ANALYST:")
    print(analysis.content[:300], "...\n")
    
    # Writer (using analyst's output)
    explanation = await writer.run(
        f"Explain this to a general audience: {analysis.content}",
        thread_id="writer-demo"
    )
    print("✍️  WRITER:")
    print(explanation.content[:300], "...\n")
    
    # Calculator
    calculation = await calculator.run(
        "Calculate EI benefits: $30,000 earnings over 20 weeks, Region 5",
        thread_id="calc-demo"
    )
    print("🧮 CALCULATOR:")
    print(calculation.content[:300], "...\n")

await test_agents()

## 2. Sequential Multi-Agent Workflow

Chain agents together for complex workflows.

In [ ]:
from agent_framework import Graph

# Build sequential graph
graph = Graph()

# Define flow: Analyst → Writer
graph.add_edge(analyst, writer)

print("✅ Sequential graph created: Analyst → Writer")

In [ ]:
# Execute sequential workflow
async def run_sequential_workflow():
    initial_state = {
        "question": "What happens if I quit my job voluntarily?",
        "context": "Employment Insurance Act, Section 30"
    }
    
    final_state = await graph.run(initial_state)
    
    return final_state

result = await run_sequential_workflow()

print("📊 WORKFLOW RESULTS:\n")
print("1️⃣  Analyst Output (first 200 chars):")
print(result.get("analyst_output", "N/A")[:200], "...\n")

print("2️⃣  Writer Output (first 200 chars):")
print(result.get("writer_output", "N/A")[:200], "...\n")

## 3. Parallel Multi-Agent Workflow

Run multiple agents simultaneously for efficiency.

In [ ]:
# Create additional specialized agents
policy_agent = Agent(
    name="PolicyResearcher",
    instructions="Research EI policy sections and regulations",
    model="gpt-4o",
    temperature=0.2
)

precedent_agent = Agent(
    name="PrecedentFinder",
    instructions="Find relevant case law and tribunal decisions",
   model="gpt-4o",
    temperature=0.3
)

calculate_agent = Agent(
    name="BenefitCalculator",
    instructions="Calculate benefit amounts and duration",
    model="gpt-4o",
    temperature=0.1
)

# Build parallel graph
parallel_graph = Graph()
parallel_graph.add_parallel([policy_agent, precedent_agent, calculate_agent])

print("✅ Parallel graph created: 3 agents run simultaneously")

In [ ]:
# Execute parallel workflow
import time

async def run_parallel_workflow():
    state = {
        "claim_scenario": "650 hours worked, Region 7, laid off, $25k earnings over 18 weeks"
    }
    
    start = time.time()
    result = await parallel_graph.run(state)
    duration = time.time() - start
    
    return result, duration

parallel_result, parallel_time = await run_parallel_workflow()

print(f"⚡ Parallel execution time: {parallel_time:.2f}s\n")
print("📊 Results from all 3 agents:\n")
for key, value in parallel_result.items():
    if isinstance(value, str) and len(value) > 100:
        print(f"{key}: {value[:150]}...\n")
    else:
        print(f"{key}: {value}\n")

## 4. Conditional Routing

Route to different agents based on conditions.

In [ ]:
# Define routing function
def route_by_claim_type(state: Dict[str, Any]) -> str:
    """
    Determine which agent to use based on claim type.
    """
    claim_type = state.get("claim_type", "regular").lower()
    
    if "regular" in claim_type:
        return "regular_evaluator"
    elif "sickness" in claim_type:
        return "sickness_evaluator"
    elif "maternity" in claim_type or "parental" in claim_type:
        return "maternity_evaluator"
    else:
        return "general_evaluator"

# Create specialized evaluators
regular_evaluator = Agent(
    name="RegularBenefitsEvaluator",
    instructions="Evaluate regular EI benefit claims",
    model="gpt-4o"
)

sickness_evaluator = Agent(
    name="SicknessBenefitsEvaluator",
    instructions="Evaluate EI sickness benefit claims (medical required)",
    model="gpt-4o"
)

maternity_evaluator = Agent(
    name="MaternityParentalEvaluator",
    instructions="Evaluate maternity/parental EI claims",
    model="gpt-4o"
)

general_evaluator = Agent(
    name="GeneralEvaluator",
    instructions="Evaluate general EI inquiries",
    model="gpt-4o"
)

# Build conditional graph
conditional_graph = Graph()
conditional_graph.add_conditional_edge(
    condition=route_by_claim_type,
    branches={
        "regular_evaluator": regular_evaluator,
        "sickness_evaluator": sickness_evaluator,
        "maternity_evaluator": maternity_evaluator,
        "general_evaluator": general_evaluator
    }
)

print("✅ Conditional graph created with 4 routing options")

In [ ]:
# Test conditional routing with different claim types
async def test_conditional_routing():
    test_cases = [
        {"claim_type": "regular", "description": "Layoff from job"},
        {"claim_type": "sickness", "description": "Unable to work due to illness"},
        {"claim_type": "maternity", "description": "Pregnancy leave"}
    ]
    
    for i, case in enumerate(test_cases, 1):
        result = await conditional_graph.run(case)
        print(f"\n{i}. Claim Type: {case['claim_type'].upper()}")
        print(f"   Routed to: {route_by_claim_type(case)}")
        print(f"   Result (first 150 chars): {str(result)[:150]}...")

await test_conditional_routing()

## 5. Complex Workflow: Full Claim Processing

Combine sequential, parallel, and conditional patterns.

In [ ]:
# Define complete workflow agents
intake_agent = Agent(
    name="IntakeProcessor",
    instructions="""Extract and validate claim information.
    Check for: applicant ID, hours worked, earnings, separation reason, region.""",
    model="gpt-4o"
)

eligibility_agent = Agent(
    name="EligibilityValidator",
    instructions="""Validate EI eligibility:
    - Minimum hours (varies by region)
    - Valid separation reason
    - Availability for work""",
    model="gpt-4o"
)

benefits_agent = Agent(
    name="BenefitsCalculator",
    instructions="""Calculate benefit amount and duration.
    Use 55% of average weekly earnings, max $668/week.""",
    model="gpt-4o"
)

approval_agent = Agent(
    name="FinalApprover",
    instructions="""Review and approve/deny claim.
    Verify all steps completed correctly.""",
    model="gpt-4o"
)

# Build comprehensive graph
full_workflow = Graph()

# Sequential main flow
full_workflow.add_edge(intake_agent, eligibility_agent)
full_workflow.add_edge(eligibility_agent, benefits_agent)
full_workflow.add_edge(benefits_agent, approval_agent)

print("✅ Full workflow graph created:")
print("   Intake → Eligibility → Benefits → Approval")

In [ ]:
# Execute full workflow
async def process_full_claim():
    claim_data = {
        "applicant_id": "EI-2026-12345",
        "applicant_name": "Jane Smith",
        "hours_worked": 720,
        "region": "Region 7 (Ottawa)",
        "unemployment_rate": "6.5%",
        "separation_reason": "Laid off - company downsizing",
        "earnings_details": "$32,000 over 22 weeks",
        "availability": "Available for full-time work, actively searching"
    }
    
    print("🔄 Processing claim through full workflow...\n")
    
    start = time.time()
    result = await full_workflow.run(claim_data)
    duration = time.time() - start
    
    return result, duration

full_result, workflow_time = await process_full_claim()

print(f"\n⏱️  Total workflow time: {workflow_time:.2f}s\n")
print("📋 WORKFLOW STAGES:\n")

stages = ["intake_output", "eligibility_output", "benefits_output", "approval_output"]
stage_names = ["Intake", "Eligibility", "Benefits", "Approval"]

for stage, name in zip(stages, stage_names):
    output = full_result.get(stage, "N/A")
    if isinstance(output, str):
        print(f"{name}:")
        print(f"{output[:200]}...\n")

## 6. Thread Management

Manage conversation history and context.

In [ ]:
from agent_framework import Thread

# Create thread with metadata
thread = Thread(
    id="user-456-session-1",
    metadata={
        "user_id": "user-456",
        "session_start": "2026-02-18T08:00:00Z",
        "channel": "web",
        "language": "en"
    }
)

print(f"✅ Thread created: {thread.id}")
print(f"   Metadata: {json.dumps(thread.metadata, indent=2)}")

In [ ]:
# Multi-turn conversation with thread
async def multi_turn_conversation():
    thread_id = "tutorial-conversation"
    
    # Turn 1
    r1 = await analyst.run("What is EI?", thread_id=thread_id)
    print("User: What is EI?")
    print(f"Agent: {r1.content[:150]}...\n")
    
    # Turn 2 (agent remembers context)
    r2 = await analyst.run("How do I qualify?", thread_id=thread_id)
    print("User: How do I qualify?")
    print(f"Agent: {r2.content[:150]}...\n")
    
    # Turn 3
    r3 = await analyst.run("What if I quit?", thread_id=thread_id)
    print("User: What if I quit?")
    print(f"Agent: {r3.content[:150]}...\n")
    
    return thread_id

conversation_thread = await multi_turn_conversation()

In [ ]:
# Access thread history
thread_obj = analyst.get_thread(conversation_thread)

if thread_obj:
    messages = thread_obj.get_messages()
    print(f"📜 Thread history ({len(messages)} messages):\n")
    
    for i, msg in enumerate(messages, 1):
        role = msg.role.upper()
        content = msg.content[:100] if len(msg.content) > 100 else msg.content
        print(f"{i}. {role}: {content}...\n")
else:
    print("⚠️  Thread history not available (framework limitation)")

## 7. Error Handling

Handle agent errors gracefully.

In [ ]:
from agent_framework import AgentError

async def safe_agent_call(agent: Agent, message: str, thread_id: str = None):
    """
    Call agent with error handling.
    """
    try:
        response = await agent.run(message, thread_id=thread_id)
        return {"success": True, "response": response.content}
        
    except AgentError as e:
        print(f"❌ Agent error: {e}")
        return {"success": False, "error": str(e), "error_type": "AgentError"}
        
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        return {"success": False, "error": str(e), "error_type": "UnexpectedError"}

# Test error handling
result = await safe_agent_call(
    analyst,
    "What are EI requirements?",
    thread_id="error-test"
)

if result["success"]:
    print(f"✅ Success: {result['response'][:100]}...")
else:
    print(f"❌ Failed: {result['error']} ({result['error_type']})")

## 8. Summary

### What You've Learned

1. ✅ **Specialized Agents:** Created agents optimized for specific tasks
2. ✅ **Sequential Workflows:** Chained agents for multi-stage processing
3. ✅ **Parallel Execution:** Ran multiple agents simultaneously
4. ✅ **Conditional Routing:** Directed flow based on logic
5. ✅ **Complex Workflows:** Built full claim processing pipeline
6. ✅ **Thread Management:** Maintained conversation context
7. ✅ **Error Handling:** Handled agent failures gracefully

### Key Patterns

- **Temperature Selection:** 0.1-0.2 for factual, 0.6-0.8 for creative
- **Graph Types:** Sequential (edges), Parallel (add_parallel), Conditional (add_conditional_edge)
- **State Management:** Shared state dictionary across agents
- **Thread IDs:** Maintain conversation history

### Next Steps

- **Notebook 03:** RAG pipelines with retrieval and generation
- **Notebook 04:** Evaluation and quality metrics
- **Production:** Deploy to Azure AI Foundry

## 9. Cleanup

In [ ]:
print("🎉 Agent Framework tutorial complete!")
print("\n📚 You're now ready to build production multi-agent systems.")